# SimSat Gemma 4 E2B Fine-tune

Fine-tunes `google/gemma-4-e2b-it` on the SimSat encounter-triage task  
(`accept | defer | refine | skip` over Sentinel satellite tiles).

**This is NOT the v35-gov governance model.** That model trains on human-interview /  
consent prompts — wrong task shape for satellite imagery triage.

## Inputs (upload to Drive before running)
```
MyDrive/simsat/
  simsat_format_train.jsonl      # 66 rows, text-only, format exposure
  simsat_multimodal_reviewed.jsonl  # 4 rows, operator-reviewed, weight=6-8
  simsat_multimodal_weak.jsonl   # 52 rows, clip_local-backed, weight=2
  images/                        # 79 PNG tiles
```
Generate locally with:
```bash
python scripts/export_gemma4_v3_mix.py --output-dir exports/gemma4_v3
```

## Output
LoRA adapter saved to `MyDrive/simsat/output/simsat-gemma4-v1/adapter/`.  
Copy locally to `weights/simsat-gemma4-v1/adapter/` and set:  
```
OBSERVATION_VLA_BACKEND=gemma4_haic_local
HAIC_GEMMA4_LORA_PATH=./weights/simsat-gemma4-v1/adapter
HAIC_GEMMA4_BASE_MODEL=google/gemma-4-e2b-it
```

In [ ]:
# ── 1. GPU PREFLIGHT ──────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'No GPU detected')

# Warn if P100 (sm_60 — not supported by modern PyTorch)
if 'P100' in result.stdout or 'sm_60' in result.stdout:
    print('⚠️  P100 detected — CUDA sm_60 is not supported. Change runtime to T4/V100/A100.')
    sys.exit(1)
print('✓ GPU OK')

In [ ]:
# ── 2. INSTALL UNSLOTH ────────────────────────────────────────────────────────
# Pin versions that work with Gemma 4 E2B
!pip install -q "unsloth[colab-new]" "trl>=0.18.2,<=0.24.0" "transformers>=4.51.3,<=5.5.0" 2>&1 | tail -5
print('✓ Unsloth installed')

In [ ]:
# ── 3. MOUNT DRIVE & SET PATHS ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATA_DIR  = Path('/content/drive/MyDrive/simsat')
OUT_DIR   = DATA_DIR / 'output' / 'simsat-gemma4-v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FORMAT_TRAIN   = DATA_DIR / 'simsat_format_train.jsonl'
REVIEWED_TRAIN = DATA_DIR / 'simsat_multimodal_reviewed.jsonl'
WEAK_TRAIN     = DATA_DIR / 'simsat_multimodal_weak.jsonl'
EVAL_FILE      = DATA_DIR / 'simsat_eval_reviewed.jsonl'
IMAGES_DIR     = DATA_DIR / 'images'

for p in [FORMAT_TRAIN, REVIEWED_TRAIN, WEAK_TRAIN]:
    assert p.exists(), f'Missing: {p}'
print('✓ Data files found')

In [ ]:
# ── 4. LOAD & INSPECT DATA ────────────────────────────────────────────────────
import json
from collections import Counter

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

format_rows   = load_jsonl(FORMAT_TRAIN)
reviewed_rows = load_jsonl(REVIEWED_TRAIN)
weak_rows     = load_jsonl(WEAK_TRAIN)
eval_rows     = load_jsonl(EVAL_FILE) if EVAL_FILE.exists() else []

for name, rows in [('format_train', format_rows), ('reviewed', reviewed_rows),
                   ('weak', weak_rows), ('eval', eval_rows)]:
    actions = Counter(r['target_json']['recommended_action'] for r in rows)
    print(f'{name:20s} {len(rows):3d} rows | {dict(actions)}')

In [ ]:
# ── 5. CONVERT TO CHATML FORMAT ───────────────────────────────────────────────
import base64
from PIL import Image
import io

SYSTEM_MSG = (
    "You are SimSat-VLA, an Earth-observation triage model. "
    "Given satellite imagery metadata and optionally a tile image, "
    "assess the encounter window and return a single JSON object. "
    "Action vocabulary: accept | defer | refine | skip."
)

SCHEMA_HINT = (
    "Respond ONLY with a JSON object — no markdown, no prose:\n"
    '{"usable_observation":<bool>,"scene_match_score":<0-1>,'
    '"salience_score":<0-1>,"change_or_event_score":<0-1>,'
    '"occlusion_or_cloud_risk":<0-1>,"confidence":<0-1>,'
    '"recommended_action":"accept|defer|refine|skip","rationale_tags":[...]}'
)

def row_to_target_json(row):
    tj = row['target_json']
    # Prefer operator action if present
    labels = row.get('review_labels') or {}
    action = labels.get('operator_action') or tj.get('recommended_action', 'refine')
    usable = labels.get('useful') if labels.get('useful') is not None else tj.get('usable_observation', False)
    return json.dumps({
        'usable_observation': bool(usable),
        'scene_match_score': round(float(tj.get('scene_match_score', 0.5)), 4),
        'salience_score': round(float(tj.get('salience_score', 0.5)), 4),
        'change_or_event_score': round(float(tj.get('change_or_event_score', 0.5)), 4),
        'occlusion_or_cloud_risk': round(float(tj.get('occlusion_or_cloud_risk', 0.35)), 4),
        'confidence': round(float(tj.get('confidence', 0.5)), 4),
        'recommended_action': action,
        'rationale_tags': [str(t) for t in tj.get('rationale_tags', [])][:6],
    })

def row_to_user_content(row, include_image=True):
    """Build user message content (text + optional image)."""
    content = []
    # Image block — load from images/ dir if available
    img_path = row.get('image_path')
    if include_image and img_path:
        full_img = IMAGES_DIR / Path(img_path).name
        if full_img.exists():
            img = Image.open(full_img).convert('RGB').resize((224, 224))
            content.append({'type': 'image', 'image': img})
    content.append({'type': 'text', 'text': row['prompt_text'] + '\n\n' + SCHEMA_HINT})
    return content

def rows_to_conversations(rows, include_images=True):
    convs = []
    for row in rows:
        user_content = row_to_user_content(row, include_image=include_images)
        convs.append({
            'messages': [
                {'role': 'system', 'content': SYSTEM_MSG},
                {'role': 'user', 'content': user_content},
                {'role': 'assistant', 'content': row_to_target_json(row)},
            ],
            'training_weight': float(row.get('training_weight', 1.0)),
        })
    return convs

# Text-only format rows (no images — just schema exposure)
format_convs   = rows_to_conversations(format_rows,   include_images=False)
reviewed_convs = rows_to_conversations(reviewed_rows, include_images=True)
weak_convs     = rows_to_conversations(weak_rows,     include_images=True)

print(f'format: {len(format_convs)} | reviewed: {len(reviewed_convs)} | weak: {len(weak_convs)}')
print('Sample assistant output:', json.loads(reviewed_convs[0]['messages'][2]['content'])['recommended_action'])

In [ ]:
# ── 6. WEIGHTED DATASET ───────────────────────────────────────────────────────
# Upsample by training_weight (rounded). Reviewed cases weight=6-8, weak=2, format=0.5-1.5
import random

def expand_by_weight(convs, min_copies=1, max_copies=8):
    expanded = []
    for conv in convs:
        w = conv.get('training_weight', 1.0)
        copies = min(max_copies, max(min_copies, round(w)))
        expanded.extend([conv] * copies)
    return expanded

all_convs = (
    expand_by_weight(format_convs)    +  # format exposure
    expand_by_weight(reviewed_convs)  +  # operator-reviewed (upsampled)
    expand_by_weight(weak_convs)         # weak multimodal
)
random.seed(42)
random.shuffle(all_convs)

print(f'Total training conversations after weighting: {len(all_convs)}')
actions = Counter(
    json.loads(c['messages'][2]['content'])['recommended_action']
    for c in all_convs
)
print(f'Action distribution: {dict(actions)}')

In [ ]:
# ── 7. LOAD GEMMA 4 E2B WITH UNSLOTH ─────────────────────────────────────────
from unsloth import FastModel
import torch

BASE_MODEL = 'google/gemma-4-e2b-it'
MAX_SEQ_LEN = 1024  # SimSat prompts + JSON response fit comfortably

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,          # auto
)
print(f'✓ Loaded {BASE_MODEL}')
print(f'  dtype={model.dtype}, device={next(model.parameters()).device}')

In [ ]:
# ── 8. LORA CONFIG ────────────────────────────────────────────────────────────
model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# ── 9. FORMAT DATASET FOR UNSLOTH ─────────────────────────────────────────────
from datasets import Dataset

def apply_chat_template(conv):
    """Flatten conversation messages to a single text string for SFT."""
    # For text-only messages, collapse content lists to string
    messages = []
    for msg in conv['messages']:
        content = msg['content']
        if isinstance(content, list):
            text_parts = [b['text'] for b in content if b.get('type') == 'text']
            content = ' '.join(text_parts)
        messages.append({'role': msg['role'], 'content': content})
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

texts = [apply_chat_template(c) for c in all_convs]
dataset = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(dataset)} examples')
print('Sample (first 300 chars):', dataset[0]['text'][:300])

In [ ]:
# ── 10. TRAIN ─────────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

ADAPTER_OUT = str(OUT_DIR / 'adapter')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch = 8
        warmup_ratio=0.1,
        num_train_epochs=4,              # small dataset — more epochs
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        output_dir=str(OUT_DIR / 'checkpoints'),
        report_to='none',
        save_strategy='epoch',
        save_total_limit=2,
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'✓ Training complete: {trainer_stats.metrics}')

In [ ]:
# ── 11. EVAL ON REVIEWED SET ──────────────────────────────────────────────────
import re

FastModel.for_inference(model)

def run_eval(row):
    prompt = row['prompt_text'] + '\n\n' + SCHEMA_HINT
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=1.0,
                              pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        m = re.search(r'\{.*\}', decoded, re.DOTALL)
        parsed = json.loads(m.group(0)) if m else {}
    except Exception:
        parsed = {}
    return decoded, parsed

if eval_rows:
    print('=== Eval on reviewed cases ===')
    correct = 0
    for row in eval_rows:
        raw, parsed = run_eval(row)
        gold = (row.get('review_labels') or {}).get('operator_action') or row['target_json']['recommended_action']
        pred = parsed.get('recommended_action', 'PARSE_FAIL')
        match = pred == gold
        correct += int(match)
        print(f'  {row["target_label"]:30s} gold={gold:6s} pred={pred:6s} {"✓" if match else "✗"}')
    print(f'Action accuracy: {correct}/{len(eval_rows)}')
else:
    print('No eval file found — skipping eval')

In [ ]:
# ── 12. SAVE ADAPTER TO DRIVE ─────────────────────────────────────────────────
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f'✓ Adapter saved to {ADAPTER_OUT}')

# Save a small metadata file for tracking
import json as _json
meta = {
    'base_model': BASE_MODEL,
    'task': 'simsat_encounter_triage',
    'action_vocab': ['accept', 'defer', 'refine', 'skip'],
    'training_rows': len(all_convs),
    'lora_r': 16,
    'lora_alpha': 32,
    'epochs': 4,
    'env_vars': {
        'OBSERVATION_VLA_BACKEND': 'gemma4_haic_local',
        'HAIC_GEMMA4_BASE_MODEL': BASE_MODEL,
        'HAIC_GEMMA4_LORA_PATH': './weights/simsat-gemma4-v1/adapter',
        'HAIC_GEMMA4_MODE': 'lora',
    },
    'metrics': trainer_stats.metrics,
}
(OUT_DIR / 'training_meta.json').write_text(_json.dumps(meta, indent=2))
print('✓ Metadata written')

print()
print('=== Next steps ===')
print('1. Download adapter/ from Drive to weights/simsat-gemma4-v1/adapter/')
print('2. Set OBSERVATION_VLA_BACKEND=gemma4_haic_local')
print('3. Set HAIC_GEMMA4_LORA_PATH=./weights/simsat-gemma4-v1/adapter')
print('4. Run: python scripts/observation_vla_eval.py --inprocess')